# Mock 10 — corrected validation report

Hour-ahead consumption Ridge model. Each **Fix:** note says what changed and why.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.width", 120)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

In [2]:
df = pd.read_csv("../../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
y = df["consumption_mwh"]
frame = pd.DataFrame({
    "lag1": y.shift(1), "lag2": y.shift(2), "lag24": y.shift(24), "lag168": y.shift(168),
    "temp": df["temp_c"], "hour": df.index.hour, "price": df["price_eur_mwh"],
    "consumption": y,
}).dropna()
features = ["lag1", "lag2", "lag24", "lag168", "temp", "hour"]

**Fix 1 — one contiguous test set, no overlap.** `loc["2023-08":"2023-10"]` and `loc["2023-10":"2023-12"]`
both contain October; the mock's test set had 4,416 rows for 3,672 hours, so October counted twice
in every metric. Always check `index.is_unique` / `index.duplicated().sum()`.

In [3]:
train = frame.loc[:"2023-07"]
test = frame.loc["2023-08":]
assert test.index.is_unique and train.index.max() < test.index.min()
print(len(train), len(test), "duplicated test timestamps in the mock:",
      pd.concat([frame.loc["2023-08":"2023-10"], frame.loc["2023-10":"2023-12"]]).index.duplicated().sum())

13680 3672 duplicated test timestamps in the mock: 744


In [4]:
model = Ridge(alpha=1.0).fit(train[features], train["consumption"])
pred_train = pd.Series(model.predict(train[features]), index=train.index)
pred_test = pd.Series(model.predict(test[features]), index=test.index)
err = test["consumption"] - pred_test

**Fix 2 — report out-of-sample numbers under the out-of-sample heading.** The mock's "OOS" R²/RMSE
were computed on `train`. **Fix 3 — `r2_score(y_true, y_pred)`**: the argument order matters
(R² is not symmetric); the mock swapped it in the one cell that did use test data.

In [5]:
print(f"in-sample  R2 {r2_score(train['consumption'], pred_train):.4f}  RMSE {rmse(train['consumption'], pred_train):.1f}")
print(f"OOS        R2 {r2_score(test['consumption'], pred_test):.4f}  RMSE {rmse(test['consumption'], pred_test):.1f}")
print(f"OOS, args swapped as in the mock: R2 {r2_score(pred_test, test['consumption']):.4f}")

in-sample  R2 0.9609  RMSE 841.6
OOS        R2 0.9571  RMSE 819.6
OOS, args swapped as in the mock: R2 0.9556


**Fix 4 — beat the naive baseline.** For an hour-ahead model the relevant benchmark is persistence
(`lag1`) and same-hour-yesterday (`lag24`). Report the model next to them.

In [6]:
pd.Series({
    "naive lag1": rmse(test["consumption"], test["lag1"]),
    "naive lag24": rmse(test["consumption"], test["lag24"]),
    "Ridge": rmse(test["consumption"], pred_test),
}, name="OOS RMSE").round(1)

naive lag1     1522.2
naive lag24    1617.8
Ridge           819.6
Name: OOS RMSE, dtype: float64

**Fix 5 — "no bias" needs a standard error and a breakdown.** A mean error of ~50 MWh on ~3,700 hours
with sd ~820 has a standard error of ~13, so it is *not* indistinguishable from zero. And the overall
mean hides a large hour-of-day pattern: the model treats `hour` as a straight line, so it is biased
by several hundred MWh at particular hours.

In [7]:
se = err.std() / np.sqrt(len(err))
print(f"mean error {err.mean():.1f} MWh, s.e. {se:.1f}, t = {err.mean() / se:.1f}")
by_hour = err.groupby(err.index.hour).agg(["mean", "std"]).round(0)
by_hour.T

mean error 48.9 MWh, s.e. 13.5, t = 3.6


time,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
mean,42.0,-436.0,-85.0,-311.0,-36.0,310.0,610.0,651.0,-347.0,-304.0,...,250.0,502.0,1250.0,939.0,-180.0,-935.0,-261.0,-32.0,-525.0,24.0
std,1039.0,706.0,646.0,619.0,650.0,642.0,642.0,627.0,628.0,663.0,...,627.0,686.0,638.0,676.0,587.0,655.0,694.0,726.0,690.0,626.0


**Fix 6 — compare predictions to actuals *paired*, not sorted.** Sorting both columns independently
produces a quantile plot: any two variables with similar distributions look "perfectly tracked"
(corr 0.9997 in the mock). The paired correlation is the honest one.

In [8]:
paired = pd.DataFrame({"actual": test["consumption"], "pred": pred_test})
print("paired corr:", round(paired.corr().iloc[0, 1], 4),
      "| sorted-independently corr:", round(np.corrcoef(np.sort(paired.actual), np.sort(paired.pred))[0, 1], 4))
paired.head(8).round(0)

paired corr: 0.9784 | sorted-independently corr: 0.9997


,actual,pred
time,,
2023-08-01 00:00:00+00:00,23055.0,22597.0
2023-08-01 01:00:00+00:00,22526.0,22292.0
2023-08-01 02:00:00+00:00,21469.0,21919.0
2023-08-01 03:00:00+00:00,21174.0,20975.0
2023-08-01 04:00:00+00:00,21461.0,21180.0
2023-08-01 05:00:00+00:00,23626.0,21836.0
2023-08-01 06:00:00+00:00,26375.0,25145.0
2023-08-01 07:00:00+00:00,28606.0,28110.0


**Fix 7 — MAPE only where the denominator is well away from zero.** The mock computed MAPE for a
price model too (prices cross zero → MAPE explodes to ~20%) and then swapped the two labels in the
summary. The consumption MAPE is the only one that belongs in this report.

In [9]:
mape_cons = (err.abs() / test["consumption"]).mean() * 100
print(f"consumption MAPE: {mape_cons:.2f}%   (min consumption in test: {test['consumption'].min():.0f} MWh, safely > 0)")

consumption MAPE: 2.25%   (min consumption in test: 18794 MWh, safely > 0)


**Fix 8 — seasonality can only be assessed on months you actually hold out.** The mock's month table
mixed in-sample and OOS predictions. The honest table covers August–December only, and the report
should say that winter and spring are *not* validated.

In [10]:
err.groupby(err.index.strftime("%Y-%m")).apply(lambda e: np.sqrt((e ** 2).mean())).round(1)

time
2023-08    773.8
2023-09    793.5
2023-10    841.2
2023-11    846.2
2023-12    840.8
dtype: float64

**Fix 9 — never delete test errors after the fact.** Dropping the worst 1% of *test* hours and re-reporting
is selection on the outcome. If they were data errors, show the evidence. Here they are not: look at
what the worst hours have in common.

In [11]:
worst = test.assign(pred=pred_test, err=err).loc[err.abs().nlargest(10).index]
worst["d_lag1"] = worst["consumption"] - worst["lag1"]
worst[["consumption", "pred", "err", "hour", "temp", "d_lag1"]].round(0)

,consumption,pred,err,hour,temp,d_lag1
time,,,,,,
2023-12-08 19:00:00+00:00,33041.0,35878.0,-2837.0,19,11.0,-1738.0
2023-10-11 17:00:00+00:00,35936.0,33164.0,2772.0,17,9.0,3026.0
2023-08-27 19:00:00+00:00,29124.0,31844.0,-2720.0,19,20.0,-2430.0
2023-09-23 00:00:00+00:00,20460.0,23152.0,-2692.0,0,12.0,-3217.0
2023-11-18 22:00:00+00:00,25944.0,28609.0,-2665.0,22,5.0,-2685.0
2023-11-15 16:00:00+00:00,33394.0,30736.0,2658.0,16,10.0,2878.0
2023-11-11 19:00:00+00:00,32742.0,35326.0,-2584.0,19,6.0,-2430.0
2023-11-10 16:00:00+00:00,32873.0,30369.0,2503.0,16,9.0,2676.0
2023-08-07 06:00:00+00:00,25645.0,23148.0,2496.0,6,15.0,3221.0


In [12]:
cut = err.abs().quantile(0.99)
big = err[err.abs() > cut]
print("hours dropped in the mock:", len(big))
print("their hour-of-day distribution:")
print(big.groupby(big.index.hour).size().to_dict())
print(f"RMSE all: {rmse(test['consumption'], pred_test):.1f}   RMSE trimmed (mock): {np.sqrt((err[err.abs() <= cut] ** 2).mean()):.1f}")

hours dropped in the mock: 37
their hour-of-day distribution:
{0: 7, 1: 1, 6: 1, 7: 1, 9: 1, 16: 8, 17: 6, 19: 7, 20: 1, 21: 2, 22: 2}
RMSE all: 819.6   RMSE trimmed (mock): 787.7


**Fix 10 — residual autocorrelation and hour-of-day bias mean the model is missing structure.**
Add hour-of-day dummies and the RMSE drops substantially; that is the actionable finding the mock
waved away.

In [13]:
Xh = pd.get_dummies(frame["hour"], prefix="h", drop_first=True).astype(float)
frame2 = pd.concat([frame, Xh], axis=1)
feats2 = ["lag1", "lag2", "lag24", "lag168", "temp"] + list(Xh.columns)
m2 = Ridge(alpha=1.0).fit(frame2.loc[train.index, feats2], train["consumption"])
p2 = pd.Series(m2.predict(frame2.loc[test.index, feats2]), index=test.index)
e2 = test["consumption"] - p2
print(f"model + hour dummies: OOS RMSE {rmse(test['consumption'], p2):.1f}, resid autocorr {e2.autocorr(1):.3f}")
print(f"original model:       OOS RMSE {rmse(test['consumption'], pred_test):.1f}, resid autocorr {err.autocorr(1):.3f}")

model + hour dummies: OOS RMSE 548.7, resid autocorr -0.006
original model:       OOS RMSE 819.6, resid autocorr 0.207


**Fix 11 — a CI for RMSE must respect dependence.** The mock used ±1.96·sd/√n on squared errors as if
3,672 hours were independent. A simple block bootstrap over days gives a wider, more honest interval.

In [14]:
rng = np.random.default_rng(0)
days = err.groupby(err.index.date)
blocks = [g.values for _, g in days]
boot = []
for _ in range(500):
    pick = rng.integers(0, len(blocks), len(blocks))
    e = np.concatenate([blocks[i] for i in pick])
    boot.append(np.sqrt((e ** 2).mean()))
iid_half = 1.96 * (err ** 2).std() / np.sqrt(len(err))
print(f"iid CI (mock style): [{np.sqrt((err**2).mean() - iid_half):.1f}, {np.sqrt((err**2).mean() + iid_half):.1f}]")
print(f"daily block bootstrap 95% CI: [{np.percentile(boot, 2.5):.1f}, {np.percentile(boot, 97.5):.1f}]")

iid CI (mock style): [800.1, 838.7]
daily block bootstrap 95% CI: [797.2, 844.4]


**Fix 12 — compare coefficients on a common scale.** Raw coefficients are in MWh per unit of each
feature: −25 MWh/°C looks tiny next to 1.1 MWh/MWh but temperature varies by degrees and lag1 by
thousands of MWh. Multiply by the feature's standard deviation (or standardise first).

In [15]:
raw = pd.Series(model.coef_, index=features)
std_coef = raw * train[features].std()
pd.DataFrame({"raw": raw, "per_1sd_MWh": std_coef}).round(2).sort_values("per_1sd_MWh", key=np.abs, ascending=False)

,raw,per_1sd_MWh
lag1,1.11,4724.44
lag2,-0.51,-2175.27
lag168,0.21,877.88
lag24,0.13,559.56
temp,-24.77,-171.00
hour,2.32,16.03


**Fix 13 — one key per metric.** The mock's summary had `"RMSE (MWh)"` and `"RMSE (MWh) "` (trailing space), so two different numbers sat under what looked like the same label.

In [16]:
summary = pd.Series({
    "R2 (OOS)": round(r2_score(test["consumption"], pred_test), 4),
    "RMSE OOS (MWh)": round(rmse(test["consumption"], pred_test), 1),
    "RMSE naive lag1 (MWh)": round(rmse(test["consumption"], test["lag1"]), 1),
    "RMSE + hour dummies (MWh)": round(rmse(test["consumption"], p2), 1),
    "MAPE consumption (%)": round(mape_cons, 2),
    "Mean error (MWh)": round(err.mean(), 1),
    "Mean error t-stat": round(err.mean() / se, 1),
    "Worst hour bias (MWh)": round(by_hour["mean"].abs().max(), 0),
    "Resid. autocorr": round(err.autocorr(1), 3),
    "Months validated": "2023-08 to 2023-12 only",
}, name="value")
summary

R2 (OOS)                                      0.9571
RMSE OOS (MWh)                                 819.6
RMSE naive lag1 (MWh)                         1522.2
RMSE + hour dummies (MWh)                      548.7
MAPE consumption (%)                            2.25
Mean error (MWh)                                48.9
Mean error t-stat                                3.6
Worst hour bias (MWh)                         1250.0
Resid. autocorr                                0.207
Months validated             2023-08 to 2023-12 only
Name: value, dtype: object

## Honest conclusions

1. OOS R² and RMSE are reported from the August–December 2023 hold-out only; winter/spring are not validated.
2. The model beats persistence but has a large hour-of-day bias; adding hour dummies fixes most of it.
3. Mean error is small relative to load but statistically non-zero.
4. No test hours were removed; the largest errors are genuine ramps at specific hours, not data errors.
5. Standardised coefficients show the lags dominate; temperature is a modest secondary driver.